# Can we predict term-deposit subscription after a marketing call?

**Question:** from client and campaign attributes in `data.csv`, can we predict `y` (subscribed yes/no)?

**Plan**
1. Inspect data and class balance
2. Decide which features are legitimately available at prediction time (important: `duration`)
3. Compare a baseline, logistic regression and gradient boosting using cross-validation and a held-out test set
4. Judge the result in business terms (ranking clients to call), not just accuracy

In [ ]:
import os; os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
RS = 42
df = pd.read_csv('data.csv')
print(df.shape, 'duplicates:', df.duplicated().sum())
df.head()

## 1. Data overview

In [ ]:
print(df.isna().sum()[lambda s: s>0].to_frame('missing').assign(pct=lambda d: (d.missing/len(df)*100).round(1)))
df['y_bin'] = (df.y=='yes').astype(int)
print('\nPositive rate:', df.y_bin.mean().round(4))
df.describe().T

About 11.7% of clients subscribe, so the data are imbalanced. A model that always says "no" scores ~88% accuracy, so **accuracy is not a useful metric**. We use ROC-AUC, average precision (PR-AUC; the no-skill level is about 0.117) and top-decile lift.

`contact` and `poutcome` are mostly missing. For `poutcome` that most likely means "not previously contacted" (consistent with `pdays = -1`), so missingness carries information. I treat missing categorical values as their own `unknown` level rather than dropping rows.

In [ ]:
cats = ['job','marital','education','default','housing','loan','contact','month','poutcome']
fig, axes = plt.subplots(3,3, figsize=(15,11))
for ax,c in zip(axes.ravel(), cats):
    r = df.assign(**{c: df[c].fillna('unknown')}).groupby(c).y_bin.agg(['mean','size']).sort_values('mean')
    ax.barh(r.index, r['mean'], color='#4C78A8'); ax.axvline(df.y_bin.mean(), color='k', ls='--', lw=1)
    ax.set_title(f'Subscription rate by {c}'); ax.set_xlabel('rate')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1,4, figsize=(16,3.5))
for ax,c in zip(axes, ['age','balance','duration','campaign']):
    for v,g in df.groupby('y'): ax.hist(g[c].clip(upper=df[c].quantile(.99)), bins=40, alpha=.6, density=True, label=f'y={v}')
    ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

## 2. The `duration` problem (target leakage)

`duration` is the length of the call. It is only known **after** the call ends, and calls that end in a sale are naturally longer. It looks very predictive but cannot be used to decide *whom to call*. I therefore evaluate two scenarios:

* **A. Pre-call model**: no `duration`. This is the realistic setting for targeting clients.
* **B. With `duration`**: shown only to quantify the leakage.

`day_of_week` here holds values 1-31 (day of month) and there is no year, so I cannot do a time-based split. I use stratified random splits; results may be slightly optimistic if the campaign drifted over time.

In [ ]:
num = ['age','balance','day_of_week','campaign','pdays','previous']
df['contacted_before'] = (df.pdays>=0).astype(int)
X_all = df[num+cats+['contacted_before','duration']].copy()
X_all[cats] = X_all[cats].fillna('unknown')
y = df.y_bin
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y, test_size=.2, stratify=y, random_state=RS)
print(X_tr.shape, X_te.shape, y_tr.mean().round(4), y_te.mean().round(4))

def models(numeric):
    return {
     'Baseline (prior)': DummyClassifier(strategy='prior'),
     'Logistic regression': make_pipeline(
         ColumnTransformer([('n', StandardScaler(), numeric), ('c', OneHotEncoder(handle_unknown='ignore'), cats)]),
         LogisticRegression(max_iter=2000)),
     'Gradient boosting': make_pipeline(
         ColumnTransformer([('n','passthrough',numeric), ('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats)]),
         HistGradientBoostingClassifier(learning_rate=.05, max_iter=300, max_leaf_nodes=15, l2_regularization=1.0,
                                        early_stopping=True, random_state=RS)),
    }
scenarios = {'A: pre-call (no duration)': num+['contacted_before'], 'B: with duration': num+['contacted_before','duration']}

## 3. Cross-validated comparison (training set only, 5-fold stratified)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RS)
rows=[]
for sname, numeric in scenarios.items():
    for mname, m in models(numeric).items():
        r = cross_validate(m, X_tr[numeric+cats], y_tr, cv=cv, scoring=['roc_auc','average_precision'], n_jobs=-1)
        rows.append(dict(scenario=sname, model=mname, roc_auc=r['test_roc_auc'].mean(), roc_auc_sd=r['test_roc_auc'].std(),
                         pr_auc=r['test_average_precision'].mean(), pr_auc_sd=r['test_average_precision'].std()))
cvres = pd.DataFrame(rows).round(3); cvres

## 4. Final evaluation on the held-out test set

In [ ]:
def lift_table(y_true, p, q=10):
    d = pd.DataFrame({'y':np.asarray(y_true),'p':p}).sort_values('p', ascending=False).reset_index(drop=True)
    d['decile'] = pd.qcut(d.index, q, labels=range(1,q+1))
    t = d.groupby('decile', observed=True).y.agg(['size','sum','mean'])
    t['lift'] = t['mean']/d.y.mean(); t['cum_capture'] = t['sum'].cumsum()/d.y.sum()
    return t
fitted, test_rows = {}, []
for sname, numeric in scenarios.items():
    cols = numeric+cats
    for mname, m in models(numeric).items():
        m.fit(X_tr[cols], y_tr); p = m.predict_proba(X_te[cols])[:,1]; fitted[(sname,mname)] = (m,p,cols)
        lt = lift_table(y_te,p)
        test_rows.append(dict(scenario=sname, model=mname, roc_auc=roc_auc_score(y_te,p), pr_auc=average_precision_score(y_te,p),
                              top_decile_lift=lt.lift.iloc[0], top20pct_capture=lt.cum_capture.iloc[1]))
test_res = pd.DataFrame(test_rows).round(3); test_res

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4.5))
colors = {'Logistic regression':'#4C78A8','Gradient boosting':'#E45756'}
for (s,mn),(m,p,_) in fitted.items():
    if mn=='Baseline (prior)': continue
    ls = '-' if s.startswith('A') else ':'
    f,t,_ = roc_curve(y_te,p); axes[0].plot(f,t,ls,color=colors[mn],label=f'{mn} [{s[0]}]')
    pr,rc,_ = precision_recall_curve(y_te,p); axes[1].plot(rc,pr,ls,color=colors[mn],label=f'{mn} [{s[0]}]')
axes[0].plot([0,1],[0,1],'k--',lw=.8); axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='ROC (solid = A, dotted = B)')
axes[1].axhline(y_te.mean(),color='k',ls='--',lw=.8); axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision-recall')
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

### What does the pre-call model buy us? Decile lift (gradient boosting, scenario A)

In [ ]:
m,p,cols = fitted[('A: pre-call (no duration)','Gradient boosting')]
lt = lift_table(y_te,p); display(lt.round(3))
fig,ax = plt.subplots(figsize=(7,3.5)); ax.bar(lt.index.astype(int), lt['mean'], color='#4C78A8')
ax.axhline(y_te.mean(), color='k', ls='--', lw=1, label='overall rate'); ax.set(xlabel='Score decile (1 = highest)', ylabel='Subscription rate'); ax.legend(); plt.show()

### What drives predictions? (permutation importance on the test set, scenario A)

In [ ]:
pi = permutation_importance(m, X_te[cols], y_te, scoring='roc_auc', n_repeats=5, random_state=RS, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=cols).sort_values()
imp.tail(12).plot.barh(figsize=(7,4), color='#4C78A8', title='Drop in test ROC-AUC when feature is shuffled'); plt.show()
print('ROC-AUC of duration alone (test):', round(roc_auc_score(y_te, X_te.duration),3))

## 5. Conclusions

In [ ]:
a = test_res.set_index(['scenario','model'])
ga = a.loc[('A: pre-call (no duration)','Gradient boosting')]; gb = a.loc[('B: with duration','Gradient boosting')]
print(f"Pre-call model: ROC-AUC {ga.roc_auc}, PR-AUC {ga.pr_auc} (no-skill {y_te.mean():.3f}), top-decile lift {ga.top_decile_lift:.1f}x, top 20% captures {ga.top20pct_capture:.0%} of subscribers")
print(f"With duration:  ROC-AUC {gb.roc_auc}, PR-AUC {gb.pr_auc}, top-decile lift {gb.top_decile_lift:.1f}x")

**Reading the results** (values from the run when this notebook was written; re-run to refresh)

* **Yes, subscription is predictable, but only moderately before the call.** Without `duration`, gradient boosting reaches test ROC-AUC of about 0.80 and PR-AUC of about 0.47 (no-skill 0.117). The top-scored decile converts at about 52% versus 11.7% overall (4.4x lift), and the top 20% of clients contain about 62% of all subscribers. That is useful for prioritising call lists, but it is not precise enough to say confidently that any single client will subscribe.
* **Adding `duration` lifts ROC-AUC to about 0.93, but it is leakage** for a targeting use case, because call length is known only after the call. Duration alone has ROC-AUC of about 0.81, as good as the whole pre-call model. Do not deploy a model that depends on it to choose whom to call.
* Gradient boosting beats logistic regression (CV ROC-AUC about 0.80 vs 0.76), so there are non-linear effects and interactions worth keeping.
* The strongest pre-call drivers are contact channel (including missing), month, day of month and previous-campaign outcome, with smaller contributions from age, number of contacts and balance. Month and day effects probably reflect campaign waves, so they may not generalise to a future campaign.
* Because the classes are imbalanced, choose a decision threshold from the cost of a call versus the value of a subscription (use the precision-recall curve or lift table), not the default 0.5.

**Caveats / next steps**
* There is no year or real timestamp, so I could not validate on a later period; real-world performance may be lower if the campaign mix shifts.
* Hyperparameters were lightly set, not tuned extensively.
* If dates are available, use a time-ordered split, and calibrate probabilities if they will be used as absolute estimates.